In [0]:
# Silver Layer — Yahoo Finance Financial Statements
# Sources: bronze.yf_income_statement, bronze.yf_balance_sheet, bronze.yf_cashflow
# Output:  silver.yf_financials  (long format: one row per company/statement/metric/period)

# Imports & Spark session

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col, lit, to_date, regexp_extract, regexp_replace, expr, isnan
)
from delta.tables import DeltaTable
import re

spark = SparkSession.builder.getOrCreate()

CATALOG = "company_risk_intelligence_platform"

# Inspect one bronze table first (understand the shape)

In [0]:
# Financial statements arrive WIDE: metric names in `index`,
# period-end dates as columns like 2024_06_30_000000
df_peek = spark.read.table(f"{CATALOG}.bronze.yf_income_statement")
df_peek.printSchema()
display(df_peek.limit(5))

#  SCD Type 1 merge helper

In [0]:
def scd_merge(source_df, target_table, business_key):
    if not spark.catalog.tableExists(target_table):
        print(f"First load -> {target_table}")
        (source_df.write.format("delta")
            .mode("overwrite")
            .saveAsTable(target_table))
        print("Table created")
    else:
        print(f"Incremental SCD1 merge -> {target_table}")
        tgt = DeltaTable.forName(spark, target_table)
        cond = " AND ".join([f"t.{k} = s.{k}" for k in business_key])
        (tgt.alias("t")
            .merge(source_df.alias("s"), cond)
            .whenMatchedUpdateAll()
            .whenNotMatchedInsertAll()
            .execute())
        print("Merge completed")

# Transformation function (wide → long / unpivot)

In [0]:
def transform_financials(df, statement_type, metric_col="index"):
    """
    Reshapes a wide financial-statement table into long format.
      ticker_safe | statement_type | metric_name | period_end_date | metric_value
    """

    # 1. Extract company key (safe ticker) from the file path
    df = df.withColumn(
        "ticker_safe",
        regexp_extract(col("file_path"), r"/\d{4}/\d{2}/\d{2}/([^/]+)/", 1)
    )

    # 2. Find the period columns (start with YYYY_MM_DD)
    period_cols = [c for c in df.columns if re.match(r"^\d{4}_\d{2}_\d{2}", c)]
    if not period_cols:
        raise ValueError(f"No period columns found for {statement_type}. "
                         f"Columns seen: {df.columns}")

    # 3. Unpivot the period columns into rows using stack()
    stack_expr = "stack({n}, {pairs}) as (period_str, metric_value)".format(
        n=len(period_cols),
        pairs=", ".join([f"'{c}', cast(`{c}` as double)" for c in period_cols])
    )

    df = df.select(
        col("ticker_safe"),
        col(metric_col).alias("metric_name"),
        col("ingestion_ts"),
        expr(stack_expr)
    )

    # 4. Parse period string (2024_06_30_000000 -> 2024-06-30) into a real date
    df = (df
        .withColumn(
            "period_end_date",
            to_date(regexp_replace(expr("substring(period_str, 1, 10)"), "_", "-"))
        )
        .withColumn("statement_type", lit(statement_type))
        .drop("period_str")
                .filter(col("metric_value").isNotNull() & ~isnan(col("metric_value")))
        .dropDuplicates(["ticker_safe", "statement_type", "metric_name", "period_end_date"])
    )

    return df.select(
        "ticker_safe", "statement_type", "metric_name",
        "period_end_date", "metric_value", "ingestion_ts"
    )

#  Transform all three and union into one DataFrame

In [0]:
configs = [
    ("yf_income_statement", "income_statement"),
    ("yf_balance_sheet",    "balance_sheet"),
    ("yf_cashflow",         "cashflow"),
]

unioned = None
for bronze_tbl, stype in configs:
    print(f"Transforming {bronze_tbl} ...")
    src = spark.read.table(f"{CATALOG}.bronze.{bronze_tbl}")
    part = transform_financials(src, stype)
    print(f"  {stype}: {part.count()} rows")
    unioned = part if unioned is None else unioned.unionByName(part)

print("Total combined rows:", unioned.count())
display(unioned.limit(20))

# Load into Silver via SCD1 merge

In [0]:
target_table = f"{CATALOG}.silver.yf_financials"
business_key = ["ticker_safe", "statement_type", "metric_name", "period_end_date"]

scd_merge(unioned, target_table, business_key)

Validation

In [0]:
silver = spark.read.table(f"{CATALOG}.silver.yf_financials")

print("Companies covered:")
display(silver.select("ticker_safe").distinct().orderBy("ticker_safe"))

print("Rows per statement type:")
display(silver.groupBy("statement_type").count())

print("Sample — one company, income statement:")
display(
    silver.filter((col("ticker_safe") == "SBRY_L") &
                  (col("statement_type") == "income_statement"))
          .orderBy("metric_name", "period_end_date")
)

In [0]:
spark.sql(f"DROP TABLE IF EXISTS {CATALOG}.silver.yf_financials")
print("Dropped old table — ready for clean rebuild")

In [0]:
silver = spark.read.table(f"{CATALOG}.silver.yf_financials")

# must print 0
print("Remaining NaNs:", silver.filter(isnan(col("metric_value"))).count())

# rows per statement type
display(silver.groupBy("statement_type").count())